In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt

df = pd.read_csv('/Users/lukehull/Desktop/lhull3.github.io/_data/ChicagoHardship.csv')
df.head()


,Community Area Number,COMMUNITY AREA NAME,PERCENT OF HOUSING CROWDED,PERCENT HOUSEHOLDS BELOW POVERTY,PERCENT AGED 16+ UNEMPLOYED,PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA,PERCENT AGED UNDER 18 OR OVER 64,PER CAPITA INCOME,HARDSHIP INDEX
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0


In [62]:
df = df.rename(columns = {'COMMUNITY AREA NAME': 'Neighborhood Name',
                          'PERCENT HOUSEHOLDS BELOW POVERTY': 'Percent of Households Below Poverty', 
                          'PERCENT AGED 16+ UNEMPLOYED': 'Percent of Population Aged 16+ Unemployed', 
                          'PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA' : 'Percent of Population 25+ Without a High School Diploma',
                          'PERCENT AGED UNDER 18 OR OVER 64': 'Percent of Population Below 18 or Over 64',
                          'PERCENT OF HOUSING CROWDED': "Percent of Housing Crowded",
                          'PER CAPITA INCOME ': 'Per Capita Income',
                          'HARDSHIP INDEX': 'Calculated Hardship Index'})

df.head()

,Community Area Number,Neighborhood Name,Percent of Housing Crowded,Percent of Households Below Poverty,Percent of Population Aged 16+ Unemployed,Percent of Population 25+ Without a High School Diploma,Percent of Population Below 18 or Over 64,Per Capita Income,Calculated Hardship Index
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0


In [63]:
options=['Percent of Housing Crowded',
        'Percent of Households Below Poverty',
        'Percent of Population Aged 16+ Unemployed',
        'Percent of Population 25+ Without a High School Diploma',
        'Percent of Population Below 18 or Over 64',
        'Per Capita Income']

dropdown = alt.binding_select(options=options, name='Indicator Selection: ')

selection = alt.selection_point(fields=['Indicator'], 
                                bind=dropdown,
                                value=[{'Indicator': 'Per Capita Income'}])

#x = alt.condition(selection,
#                  alt.X('Indicator:N', axis = alt.Axis(title = 'Indicator:N')),
#                  alt.X('Per Capita Income:N', axis = alt.Axis(title = 'Per Capita Income')))

scatterplot = alt.Chart(df).mark_circle(size = 100).encode(
    x = alt.X('Value:Q', axis = alt.Axis(title = 'Selected Indicator'), scale=alt.Scale(zero=False)),
    y = alt.Y('Calculated Hardship Index:Q', axis = alt.Axis(title = 'Hardship Index'), scale=alt.Scale(domain=[0,100])),
    color = alt.Color('Calculated Hardship Index:Q').scale(scheme="blueorange").legend(None),
    tooltip = ['Neighborhood Name:N', 'Calculated Hardship Index:Q']
).transform_fold(options, as_=['Indicator', 'Value']
).transform_filter(selection
).properties(title = 'Hardship Index by Various Socioeconomic Indicators across Chicago Neighborhoods (2008-2012)',
             width = 500,
             height = 500)

trendline = alt.Chart(df).mark_line(color='darkblue', clip=True).encode(
    x=alt.X('Value:Q', scale=alt.Scale(zero=False)),
    y=alt.Y('Calculated Hardship Index:Q', scale=alt.Scale(domain=[0,100])),
    opacity = alt.value(0.5)
).transform_fold(options,
                 as_=['Indicator', 'Value']
).transform_filter(selection
).transform_regression('Value', 'Calculated Hardship Index')

chart = (scatterplot + trendline).add_params(selection)

chart

alt.LayerChart(...)

In [64]:
chart.save("/Users/lukehull/Desktop/lhull3.github.io/assets/json/mainScatter.json")